# Kinyarwanda emotion detection: Transformer fine-tuning (Colab)

Runs experiment group 3 (`src/transformer.py`) on a free Colab GPU.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

| Exp | Model | Kinyarwanda in pretraining? |
|---|---|---|
| T1 | XLM-R base | No |
| T2 | AfriBERTa large | Yes |
| T3 | AfroXLMR base | Yes (XLM-R base + African adaptation) |
| T4 | AfroXLMR large | Yes, bigger |
| T5 | AfroXLMR base + LoRA | Yes, ~0.4% of weights trained |


## 1. Get the code and data

In [ ]:
REPO_URL = "https://github.com/allia-wase/Kinyarwanda_Emotion.detection.git"  # <-- edit

import os
if not os.path.exists("Kinyarwanda_Emotion.detection"):
    !git clone {REPO_URL}
%cd Kinyarwanda_Emotion.detection
!pip install -q -r requirements.txt
# Colab ships an old torchao that recent peft refuses to load (breaks LoRA, T5); we don't use it.
!pip uninstall -q -y torchao
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!python -m src.data   # downloads BRIGHTER (kin), removes the duplicated dev/test rows

## 2. Fine-tune
Each cell runs one experiment with 3 seeds and appends a row to `results/experiments.csv`.

In [ ]:
!python -m src.transformer --only T1_xlmr_base

In [ ]:
!python -m src.transformer --only T3_afroxlmr_base

In [ ]:
!python -m src.transformer --only T5_afroxlmr_base_lora

In [ ]:
!python -m src.transformer --only T2_afriberta_large

In [ ]:
!python -m src.transformer --only T4_afroxlmr_large

## 3. Results

In [ ]:
import pandas as pd, json
df = pd.read_csv("results/experiments.csv")
cols = ["experiment", "test_macro_f1", "test_micro_f1"] + [c for c in df.columns if c.startswith("test_f1_")]
df["±std"] = df["config"].map(lambda c: json.loads(c).get("test_macro_f1_std"))
df[cols + ["±std"]].round(3)

## 4. Save everything back
Downloads results (CSV, figures, error files). Commit the `results/` folder to GitHub. The best model is uploaded to the Hugging Face Hub in step 5.

In [ ]:
!zip -qr results_transformers.zip results
from google.colab import files
files.download("results_transformers.zip")

## 5. Upload the best model to the Hugging Face Hub (for the web app)
Paste a **write** token from https://huggingface.co/settings/tokens when asked.

In [ ]:
BEST = "T4_afroxlmr_large"          # <-- the experiment with the best DEV macro-F1
HF_REPO = "Alliane/kinyarwanda-emotion-afroxlmr"   # <-- edit

from huggingface_hub import login, HfApi
login()
api = HfApi()
api.create_repo(HF_REPO, exist_ok=True)
api.upload_folder(folder_path=f"models/{BEST}", repo_id=HF_REPO)
print("uploaded: https://huggingface.co/" + HF_REPO)